# Notebook 11: Advanced Vision: Vision Transformers (ViT, Swin), CLIP & Generative Synthesis
**Course:** CSE471 — Deep Learning for Computer Vision  
**Unit:** Unit V (Vision Transformers) & Unit VI (VLMs and Generative Models) | **CO:** CO5, CO6  
**Problem Statement Context:** SIH26131 — Cutting-Edge Multi-Modal & Self-Attention Architectures  

---

## 1. Vision Transformers (ViT) from First Principles
Vision Transformers (Dosovitskiy et al., 2020) treat 2D images like 1D sequence tokens:
1. Divide image $x \in \mathbb{R}^{H 	imes W 	imes C}$ into non-overlapping patches $x_p \in \mathbb{R}^{N 	imes (P^2 C)}$, where $N = rac{HW}{P^2}$.
2. Project patches linearly into $D$-dimensional embeddings.
3. Prepend a learnable `[CLS]` token and add 1D positional encodings $E_{pos} \in \mathbb{R}^{(N+1) 	imes D}$.
4. Process through Multi-Head Self-Attention (MHSA) blocks:

$$	ext{Attention}(Q, K, V) = 	ext{softmax}\left(rac{QK^T}{\sqrt{d_k}}ight) V$$

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.vlm_and_gradcam import CropVisionLanguageMatcher

## 2. Patch Extraction & Multi-Head Self-Attention Implementation

In [ ]:
class PatchEmbedding(nn.Module):
    """Converts 2D image into 1D patch embedding vectors."""
    def __init__(self, in_channels=3, patch_size=16, embed_dim=128):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # [B, C, H, W] -> [B, D, H/P, W/P] -> [B, N, D]
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class SimpleSelfAttention(nn.Module):
    """Scaled Dot-Product Self-Attention."""
    def __init__(self, embed_dim=128, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # [B, heads, N, head_dim]

        scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v).permute(0, 2, 1, 3).reshape(B, N, D)
        return self.proj(out), attn

print("[✓] PatchEmbedding and SimpleSelfAttention compiled.")

## 3. Visualize Image Patches & Self-Attention Map

In [ ]:
sample_path = REPO_ROOT / "data" / "samples" / "classification" / "lab_plantvillage" / "Tomato___Early_Blight" / "sample_000.jpg"
img_pil = Image.open(sample_path).convert("RGB").resize((128, 128))
img_tensor = torch.from_numpy(np.array(img_pil, dtype=np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0)

patch_embed = PatchEmbedding(in_channels=3, patch_size=16, embed_dim=64)
attn_block = SimpleSelfAttention(embed_dim=64, num_heads=2)

patches = patch_embed(img_tensor)  # 8x8 = 64 patches
out, attn_weights = attn_block(patches)

print(f"Patches shape: {patches.shape} (64 patches of dim 64)")
print(f"Attention weights shape: {attn_weights.shape}")

# Plot attention matrix across patches
plt.figure(figsize=(6, 5))
plt.imshow(attn_weights[0, 0].detach().numpy(), cmap="viridis")
plt.title("Head 0 Self-Attention Weights Across Leaf Patches")
plt.xlabel("Key Patch Index")
plt.ylabel("Query Patch Index")
plt.colorbar()
plt.tight_layout()
plt.show()

## 4. Vision-Language (CLIP) Zero-Shot Prompt Matching

In [ ]:
matcher = CropVisionLanguageMatcher()

# Simulated feature embeddings for demonstration
np.random.seed(42)
leaf_feature = np.random.randn(128)
# Inject target correlation
class_embeddings = {
    cls_name: np.random.randn(128) for cls_name in matcher.descriptions.keys()
}
# Strong alignment with Early Blight description
class_embeddings["Tomato___Early_Blight"] = leaf_feature + np.random.randn(128) * 0.2

results = matcher.zero_shot_match(leaf_feature, class_embeddings)
print("CLIP Zero-Shot Ranking for Symptom Description:")
for cls_name, score in results[:3]:
    print(f"  * {cls_name:<25} Cosine Match: {score:.3f}")

## 5. Generative Models in Agriculture (CycleGAN & DCGAN)
- **CycleGAN:** Translates lab-controlled leaves (PlantVillage) into realistic field conditions (PlantDoc) without paired images, synthesizing domain diversity to train robust detectors.
- **DCGAN:** Generates synthetic lesion textures to balance rare agricultural pest classes.